In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
v1 = model.encode(q1)
v1.shape

(384,)

In [5]:
v2 = model.encode(q2)
v2.shape

(384,)

In [6]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.32332397)

In [9]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [10]:
v2.dot(dv)

np.float32(0.019730574)

In [11]:
v_king = model.encode('king')
v_queen = model.encode('queen')
v_king.dot(v_queen)

np.float32(0.6807127)

In [12]:
model.encode('apple').dot(model.encode('pear'))

np.float32(0.47442532)

In [13]:
model.encode('apple').dot(model.encode('car'))

np.float32(0.4098197)

In [14]:
model.encode('apple').dot(model.encode('iphone'))

np.float32(0.7238294)

## Embedding dataset

In [20]:
import sys
from pathlib import Path

In [21]:
sys.path.append(str(Path("..").resolve()))

In [23]:
from src.ingest import load_faq_data

In [24]:
documents = load_faq_data()
len(documents)

1406

In [25]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

len(texts)

1406

In [26]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/29 [00:00<?, ?it/s]

1406

In [27]:
import numpy as np
X = np.array(vectors)
X.shape

(1406, 384)

In [29]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

scores = X.dot(v1)
len(scores)

1406

In [31]:
top5 = np.argsort(-scores)[:5]
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [32]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

## Minsearch

In [33]:
from minsearch import VectorSearch

In [34]:
v_index = VectorSearch(keyword_fields=['course'])
v_index.fit(X, documents)

In [35]:
v_index.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project s

## RAG

In [40]:
from src.ingest import load_faq_data, build_index
from src.rag_helper import RAGBase

In [39]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

### Text search

In [41]:
documents = load_faq_data()
index = build_index(documents)

In [42]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [43]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

### Vector search

In [44]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [45]:
vector_assistant = RAGVector(
    embedder=model,
    index=v_index,
    llm_client=openai_client,
)

In [46]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

## SQLiteSearch

In [47]:
from sqlitesearch import VectorSearchIndex

In [50]:
vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [51]:
vs_index.fit(vectors, documents)

In [53]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
len(results)

5

In [54]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [55]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [56]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [57]:
vs_index.close()